# Day 13 — pydantic-settings: Config Management

**Phase 1 · Module 2: Bedrock Agents** · ~30 min

You now call Bedrock and read from S3 — both need **config**: a region, a model id, an S3 bucket, maybe an API key. The wrong way is to sprinkle those as string literals through your code. The right way is **one typed settings object**, loaded from the environment and a `.env` file, validated at startup. Today's tool: **`pydantic-settings`** — Pydantic's `BaseModel` you already know (Day 10), but it fills itself from the environment.

> **Runnable & keyless.** No real secrets. We set fake environment variables *inside* the notebook so every cell runs, and never read a real credential.


## 1. Why not just hardcode it?

A literal like `region = "eu-west-2"` buried in a function is a landmine: it's duplicated, it can't change per environment (dev vs prod), and the moment someone writes `api_key = "sk-live-..."` it ends up in Git history forever. Watch the smell first, then fix it.


In [1]:
# ❌ the anti-pattern — config scattered as literals
def call_model_bad(prompt):
    region = "eu-west-2"                       # duplicated everywhere
    model  = "eu.anthropic.claude-sonnet-4-5-20250929-v1:0"
    api_key = "sk-DO-NOT-COMMIT-THIS"          # secret in source = leaked
    return f"[{region}] {model} <- {prompt!r}"

print(call_model_bad("hello"))


[eu-west-2] eu.anthropic.claude-sonnet-4-5-20250929-v1:0 <- 'hello'


☝ Three problems in five lines: the region/model are copy-pasted into every function, there's no way to switch them for prod, and a **secret is sitting in the source file**. `pydantic-settings` removes all three by moving config *out* of code and into the environment.


## 2. A `BaseSettings` class — config as a typed object

`BaseSettings` looks like a normal Pydantic model, but each field is **auto-populated from an environment variable of the same name** (case-insensitive). Give types and defaults; Pydantic coerces and validates. First we set some fake env vars so the notebook is self-contained.


In [2]:
import os
# pretend these were exported in your shell / injected by AWS — set here so the cell runs
os.environ["AWS_REGION"]   = "eu-west-2"
os.environ["BEDROCK_MODEL"]= "eu.anthropic.claude-sonnet-4-5-20250929-v1:0"
os.environ["MAX_TOKENS"]   = "512"          # note: a *string* in the env

from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    aws_region: str = "us-east-1"            # default used only if env var absent
    bedrock_model: str
    max_tokens: int = 256                     # env string '512' -> int 512 automatically

settings = Settings()
print(settings)
print("max_tokens is", type(settings.max_tokens).__name__, "=", settings.max_tokens)


aws_region='eu-west-2' bedrock_model='eu.anthropic.claude-sonnet-4-5-20250929-v1:0' max_tokens=512
max_tokens is int = 512


☝ `MAX_TOKENS="512"` arrived from the environment as a **string** and came out as an **int** — Pydantic coerced and validated it. `aws_region` fell back to its default only because... it didn't (the env var was set). One object now holds all config, typed, with editor autocomplete.


## 3. Load from a **`.env`** file

In development you don't want to `export` a dozen vars by hand — you keep them in a **`.env`** file (git-ignored). Point `pydantic-settings` at it with `model_config`. Real env vars still **win** over the file, which is exactly what you want in production.


In [3]:
from pydantic_settings import SettingsConfigDict

# write a throwaway .env next to the notebook so this is self-contained
with open(".env.demo", "w") as f:
    f.write("S3_BUCKET=barclays-kb-docs\n")
    f.write("LANGSMITH_TRACING=true\n")

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env.demo", extra="ignore")
    aws_region: str = "us-east-1"
    s3_bucket: str
    langsmith_tracing: bool = False           # 'true' -> True

s = Settings()
print("bucket  :", s.s3_bucket)
print("tracing :", s.langsmith_tracing, "(", type(s.langsmith_tracing).__name__, ")")
print("region  :", s.aws_region, "(from real env, overriding any file value)")


bucket  : barclays-kb-docs
tracing : True ( bool )
region  : eu-west-2 (from real env, overriding any file value)


☝ `s3_bucket` came from the file; `langsmith_tracing` parsed the string `'true'` into a real `bool`. Precedence to remember: **real environment variables > `.env` file > field default**. Add `.env` to `.gitignore` and commit a `.env.example` with blank values instead.


## 4. Secrets — `SecretStr` so keys don't leak into logs

An API key is config too, but a *dangerous* kind: print it or log it by accident and it's exposed. `SecretStr` wraps it so it prints as `**********`; you call `.get_secret_value()` only at the point of use.


In [4]:
from pydantic import SecretStr

os.environ["BEDROCK_API_KEY"] = "sk-live-supersecret-123"

class Settings(BaseSettings):
    bedrock_api_key: SecretStr

cfg = Settings()
print("repr :", cfg)                          # masked
print("str  :", cfg.bedrock_api_key)          # masked
print("real :", cfg.bedrock_api_key.get_secret_value()[:7], "... (only when needed)")


repr : bedrock_api_key=SecretStr('**********')
str  : **********
real : sk-live ... (only when needed)


☝ The key is masked everywhere it might be logged, and only revealed by an explicit `.get_secret_value()`. This is the difference between a secret that lives safely in memory and one that shows up in a CloudWatch log or a stack trace.


## 5. Validate config at **startup**, fail loud

A missing required var should crash your app **on boot** with a clear message — not 20 minutes later mid-request. Because `Settings()` validates on construction, instantiating it once at startup gives you that fail-fast behaviour for free. Field validators (Day 10) add business rules.


In [5]:
from pydantic import field_validator
from pydantic_settings import SettingsConfigDict

class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="APP_")   # reads APP_* vars
    aws_region: str
    max_tokens: int = 256

    @field_validator("max_tokens")
    @classmethod
    def sane_tokens(cls, v):
        if not (1 <= v <= 4096):
            raise ValueError("max_tokens must be 1..4096")
        return v

# missing APP_AWS_REGION -> boot fails with a precise message
try:
    AppSettings()
except Exception as e:
    print("startup blocked:\n", e)


startup blocked:
 1 validation error for AppSettings
aws_region
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


☝ No region → a precise validation error naming the missing field, **before** the app serves a single request. `env_prefix="APP_"` namespaces your vars (`APP_AWS_REGION`) so they don't collide with other tools' env. Config errors become boot errors — the cheapest place to catch them.


## 6. The pattern — one shared settings instance

Create the settings object **once** and import it everywhere. Now the hardcoded function from §1 becomes clean, testable, and environment-driven.


In [6]:
class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env.demo", extra="ignore")
    aws_region: str = "us-east-1"
    bedrock_model: str = "eu.anthropic.claude-sonnet-4-5-20250929-v1:0"
    max_tokens: int = 256

settings = Settings()          # <- the single source of truth, imported by the whole app

def call_model_good(prompt, cfg=settings):
    return f"[{cfg.aws_region}] {cfg.bedrock_model} (max={cfg.max_tokens}) <- {prompt!r}"

print(call_model_good("hello"))
# clean up the demo file
os.remove(".env.demo")


[eu-west-2] eu.anthropic.claude-sonnet-4-5-20250929-v1:0 (max=512) <- 'hello'


☝ Same behaviour as §1, none of the sins: no duplicated literals, no secret in source, and you can point it at a different environment by changing env vars — code untouched. Pass `cfg` in for tests to inject a fake `Settings`.


## 7. Recap + checklist

| Need | pydantic-settings feature |
|---|---|
| config from env vars | `BaseSettings` fields auto-read env |
| dev config file | `SettingsConfigDict(env_file=".env")` |
| namespaced vars | `env_prefix="APP_"` |
| protect secrets | `SecretStr` + `.get_secret_value()` |
| fail fast on bad config | validated on `Settings()` + `@field_validator` |
| precedence | real env **>** `.env` **>** default |

**Your 30 min today**
- [ ] Write a `Settings(BaseSettings)` with region, model, max_tokens.
- [ ] Load one value from a `.env` file, override it with a real env var.
- [ ] Wrap a key in `SecretStr`; prove it prints masked.
- [ ] **Refactor any hardcoded config in your existing code into Settings today.**

